In [4]:
import ast
import json
import re
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.normalizers import NFKC, Sequence
from transformers import PreTrainedTokenizerFast
import numpy as np
from transformers import AutoTokenizer, PreTrainedTokenizerFast

In [5]:
def parse_options(opt_data):
    """Convert dictionary string or Python structure into clean text."""
    if not opt_data:
        return ""
    if isinstance(opt_data, dict):
        return " . ".join(opt_data.values())
    if isinstance(opt_data, list):
        return " . ".join(opt_data)
    # If stored as a dictionary string:
    try:
        parsed = ast.literal_eval(opt_data)
        if isinstance(parsed, dict):
            return " . ".join(parsed.values())
    except Exception:
        pass
    # If structural parsing fails, strip JSON symbols and digits using regex:
    cleaned = re.sub(r"[{}\"':\d]", " ", str(opt_data))
    return re.sub(r"\s+", " ", cleaned).strip()

print("Extracting text data...")
dataset = load_dataset("MohammadJRanjbar/PersianMedQA", split="train")

output_file = "medical_train_clean.txt"

with open(output_file, "w", encoding="utf-8") as f:
    for row in dataset:
        q_fa = (row.get("question") or "").strip().replace("\n", " ")
        a_fa = parse_options(row.get("answer"))
        
        q_en = (row.get("question_english") or "").strip().replace("\n", " ")
        a_en = parse_options(row.get("answer_english"))
        
        # Write independent lines with clear boundaries
        for text in [q_fa, a_fa, q_en, a_en]:
            if text:
                f.write(text + "\n")

print(f"Cleaning process completed. Output saved to: {output_file}")

Extracting text data...


Using the latest cached version of the dataset since MohammadJRanjbar/PersianMedQA couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\sajjad\.cache\huggingface\datasets\MohammadJRanjbar___persian_med_qa\default\0.0.0\286080bd0092be5a708bc7dcedacc9e32aea2b57 (last modified on Sun Sep 20 12:05:09 2026).


Cleaning process completed. Output saved to: medical_train_clean.txt


In [6]:
input_file = "legal_train_60mb.txt"
output_clean_file = "legal_train_clean.txt"

# Patterns to match ISO timestamps and long identifiers
# Example: 2024-11-11T21:45:15.999Z and metadata digits
iso_timestamp_pattern = r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d+)?Z?"
id_pattern = r"^\s*\d{5,}\s*"

with open(input_file, "r", encoding="utf-8") as infile, open(output_clean_file, "w", encoding="utf-8") as outfile:
    for line in infile:
        # 1. Remove log timestamps
        line = re.sub(iso_timestamp_pattern, "", line)
        
        # 2. Remove multi-digit database identifiers
        line = re.sub(r"\b\d{6,}\b", "", line)
        
        # 3. Normalize multiple whitespaces
        line = re.sub(r"\s+", " ", line).strip()
        
        # 4. Filter out very short or non-sentential lines
        if len(line.split()) >= 4:
            outfile.write(line + "\n")

print(f"Legal data cleaning completed. Output saved to: {output_clean_file}")

Legal data cleaning completed. Output saved to: legal_train_clean.txt


In [7]:
with open("bilingual_med_legal_corpus.txt", "w", encoding="utf-8") as final_corpus:
    for src in ["medical_train_clean.txt", "legal_train_clean.txt"]:
        with open(src, "r", encoding="utf-8") as f:
            lines = f.readlines()
            print(f"{src}: {len(lines)} خط")
            for line in lines:
                final_corpus.write(line)
        final_corpus.write("\n")

print("Unified corpus is ready for use.")

with open("bilingual_med_legal_corpus.txt", "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)
print(f"مجموع کل خطوط پیکره: {total_lines}")

medical_train_clean.txt: 58190 خط
legal_train_clean.txt: 84772 خط
Unified corpus is ready for use.
مجموع کل خطوط پیکره: 142964


In [8]:
print("1. Configuring tokenizer architecture...")
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.normalizer = Sequence([NFKC()])
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

# 2. Define Transformer control tokens
special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask()>", "<sep>"]

trainer = BpeTrainer(
    vocab_size=24000,
    min_frequency=2,
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)

print("2. Training on multi-domain corpus...")
tokenizer.train(["bilingual_med_legal_corpus.txt"], trainer)

# 3. Save in native JSON format
tokenizer.save("med_legal_tokenizer.json")

# 4. Wrap in Hugging Face PreTrainedTokenizerFast class for model integration
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="med_legal_tokenizer.json",
    pad_token="<pad>",
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>",
    mask_token="<mask()>",
    sep_token="<sep>"
)

fast_tokenizer.save_pretrained("./custom_med_legal_tokenizer")
print("3. Custom tokenizer successfully trained and saved!")

1. Configuring tokenizer architecture...
2. Training on multi-domain corpus...
3. Custom tokenizer successfully trained and saved!


In [9]:
print("--- Loading tokenizers for comparative benchmarking ---")

# 1. Custom tokenizer
custom_tokenizer = PreTrainedTokenizerFast.from_pretrained("./custom_med_legal_tokenizer")

# 2. Industry reference tokenizers (requires internet access)
# If LLaMA 3 is not accessible, fall back to multilingual models like XLM-RoBERTa or mBERT
reference_models = {
    "Ours (Custom Med-Legal)": custom_tokenizer,
}

try:
    reference_models["LLaMA 3 (General SOTA)"] = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
except Exception:
    print("LLaMA tokenizer not accessible; using XLM-RoBERTa as multilingual fallback...")
    reference_models["XLM-RoBERTa (Multilingual)"] = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 3. Challenging test samples (clinical cases, code-switching, and formal penal texts)
benchmark_corpus = [
    "بیمار با علائم تنگی نفس حاد ناشی از انسداد عروق کرونر تحت آنژیوپلاستی اورژانسی قرار گرفت.",
    "مطابق ماده ۲۲ قانون ثبت اسناد و املاک، تصرف به عنوان مالکیت دلیل بر حق مالکیت است.",
    "در آزمایش CBC بیمار مقادیر MCV=67fl و MCH=21 گزارش شده و سوفل قلبی سیستولیک شنیده شد.",
    "خواهان با ارائه دادخواست اعسار از پرداخت محکوم‌به، تقاضای تقسیط دیون قانونی خود را دارد.",
    "A 9-month-old male infant is admitted due to intermittent irritability accompanied by bilious vomiting."
]

print("\n" + "="*85)
print(f"{'Tokenizer Model':<30} | {'Total Tokens':<18} | {'Avg Fertility (Tokens/Word)':<25}")
print("="*85)

for name, tok in reference_models.items():
    total_tokens = 0
    total_words = 0
    
    for sentence in benchmark_corpus:
        words = sentence.split()
        # Encode and retrieve token IDs
        tokens = tok.encode(sentence)
        total_tokens += len(tokens)
        total_words += len(words)
        
    fertility = total_tokens / total_words
    print(f"{name:<30} | {total_tokens:<18} | {fertility:<25.2f}")

print("="*85)

--- Loading tokenizers for comparative benchmarking ---
LLaMA tokenizer not accessible; using XLM-RoBERTa as multilingual fallback...

Tokenizer Model                | Total Tokens       | Avg Fertility (Tokens/Word)
Ours (Custom Med-Legal)        | 100                | 1.28                     
XLM-RoBERTa (Multilingual)     | 141                | 1.81                     


In [10]:
sample_phrase = "درخواست اعسار و آنژیوپلاستی عروق"
tokens = custom_tokenizer.tokenize(sample_phrase)
print(f"\nTest text: {sample_phrase}")
print(f"Tokenized subwords by your model: {tokens}")
print(f"Total token count: {len(tokens)}")


Test text: درخواست اعسار و آنژیوپلاستی عروق
Tokenized subwords by your model: ['Ø¯Ø±Ø®ÙĪØ§Ø³Øª', 'ĠØ§Ø¹Ø³Ø§Ø±', 'ĠÙĪ', 'ĠØ¢ÙĨÚĺÛĮ', 'ÙĪÙ¾ÙĦØ§Ø³ØªÛĮ', 'ĠØ¹Ø±ÙĪÙĤ']
Total token count: 6
